Create an example noteboook where you connect to azure and create plots for the budget status

In [ ]:
# Azure connections

import logging
import os
import io
from datetime import datetime
import requests
import msal
import pandas as pd
import azure.functions as func

import yaml

from auth_helper import get_access_token

import requests
import urllib.parse

from datetime import datetime

In [ ]:
from dotenv import load_dotenv
load_dotenv()
client_id = os.getenv("CLIENT_ID")

In [ ]:
# Load credentials from YAML file
with open('certifications.yaml', 'r') as file:
    config = yaml.safe_load(file)

CLIENT_ID = config['client_id']
CLIENT_SECRET= config['client_secret']
TENANT_ID = config['tenant_id']
SCOPES = config['scopes']

# Obtain access token
access_token = get_access_token(CLIENT_ID, TENANT_ID, SCOPES)

# Set up headers
headers = {
    'Authorization': f'Bearer {access_token}'
}

# OneDrive API endpoint
#drive_api_endpoint = 'https://graph.microsoft.com/v1.0/me/drive/root/children'

subfolder_path = 'Documentos/PERSONAL FINANCES/money_manager_files'#2024-11-14.xlsx'
budget_file_path = 'Documentos/PERSONAL FINANCES/budget_files'
budget_status_path = 'Documentos/PERSONAL FINANCES/budget_status_files'

# URL-encode the subfolder path
encoded_subfolder_path = urllib.parse.quote(budget_status_path)
drive_api_endpoint = f'https://graph.microsoft.com/v1.0/me/drive/root:/{encoded_subfolder_path}:/children'


# Get the list of files
response = requests.get(drive_api_endpoint, headers=headers)
if response.status_code == 200:
    files_data = response.json()
    files = files_data.get('value', [])
    # Print file names
    print("Files in OneDrive Root Directory:")
    for file in files:
        print(f"{file['name']} - Last Modified: {file['lastModifiedDateTime']}")
else:
    print(f"Error retrieving files: {response.status_code}")
    print(response.json())

In [ ]:
# Load credentials from YAML file
with open('certifications.yaml', 'r') as file:
    config = yaml.safe_load(file)

CLIENT_ID = config['client_id']
CLIENT_SECRET= config['client_secret']
TENANT_ID = config['tenant_id']
SCOPES = config['scopes']

# Obtain access token
access_token = get_access_token(CLIENT_ID, TENANT_ID, SCOPES)

# Set up headers
headers = {
    'Authorization': f'Bearer {access_token}'
}

# OneDrive API endpoint
#drive_api_endpoint = 'https://graph.microsoft.com/v1.0/me/drive/root/children'

subfolder_path = 'Documentos/PERSONAL FINANCES/money_manager_files'#2024-11-14.xlsx'
budget_file_path = 'Documentos/PERSONAL FINANCES/budget_files'
budget_status_path = 'Documentos/PERSONAL FINANCES/budget_status_files'

# URL-encode the subfolder path
encoded_subfolder_path = urllib.parse.quote(budget_status_path)
drive_api_endpoint = f'https://graph.microsoft.com/v1.0/me/drive/root:/{encoded_subfolder_path}:/children'


# Get the list of files
response = requests.get(drive_api_endpoint, headers=headers)
if response.status_code == 200:
    files_data = response.json()
    files = files_data.get('value', [])
    # Print file names
    print("Files in OneDrive Root Directory:")
    for file in files:
        print(f"{file['name']} - Last Modified: {file['lastModifiedDateTime']}")
else:
    print(f"Error retrieving files: {response.status_code}")
    print(response.json())

def get_files_and_dates(folder_path):
    encoded_folder_path = urllib.parse.quote(folder_path)
    drive_api_endpoint = f'https://graph.microsoft.com/v1.0/me/drive/root:/{encoded_folder_path}:/children'
    # Get the list of files
    response = requests.get(drive_api_endpoint, headers=headers)
    if response.status_code == 200:
        files_data = response.json()
        files = files_data.get('value', [])
        # Print file names
        print("Files in OneDrive Root Directory:")
        for file in files:
            print(f"{file['name']} - Last Modified: {file['lastModifiedDateTime']}")
    else:
        print(f"Error retrieving files: {response.status_code}")
        print(response.json())

    return files

def get_latest_file(files):
    latest_file = None
    latest_time = None
    for file in files:
        file_time = datetime.strptime(file['lastModifiedDateTime'], '%Y-%m-%dT%H:%M:%SZ')
        if not latest_time or file_time > latest_time:
            latest_time = file_time
            latest_file = file
    return latest_file

def download_file(file, headers):
    file_id = file['id']
    download_url = f"https://graph.microsoft.com/v1.0/me/drive/items/{file_id}/content"
    response = requests.get(download_url, headers=headers)
    if response.status_code == 200:
        filename = file['name']
        with open(filename, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded {filename}")
        return filename
    else:
        print(f"Failed to download {file['name']}")
        return None


files_budget = get_files_and_dates('Documentos/PERSONAL FINANCES/budget_status_files')
latest_budget_file = get_latest_file(files_budget)